# Notebook 4: Mineral Deposit Clustering via Semantic Embeddings

This notebook clusters countries by their **critical mineral production profiles** using a
state-of-the-art open-source embedding model combined with dimensionality reduction and
density-based clustering.

**Pipeline overview:**
1. Load BGS production data and build a natural-language profile for each producing country.
2. Encode profiles with **BAAI/bge-large-en-v1.5** — a 1024-dimensional bi-encoder that runs
   fully locally (no API key required). It consistently ranks near the top of the MTEB
   retrieval benchmarks and captures nuanced semantic relationships between production profiles.
3. Reduce the 1024-D embedding space to 2-D with **UMAP** (Uniform Manifold Approximation and
   Projection), which preserves both local neighbourhood structure and global topology better
   than PCA or t-SNE for this kind of high-dimensional data.
4. Identify clusters with **HDBSCAN** (Hierarchical Density-Based Spatial Clustering), which
   discovers arbitrarily shaped clusters and explicitly labels outlier countries as noise.
5. Explore results interactively with Plotly and derive actionable geopolitical insights.

## 1. Setup

In [ ]:
import pathlib
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
import umap
import hdbscan

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = pathlib.Path("__file__").resolve().parent.parent  # notebooks/../
DATA_DIR  = REPO_ROOT / "data" / "bgs_data"
CSV_PATH  = DATA_DIR / "bgs_critical_minerals_production.csv"

# Fall back to a relative path when running from the notebooks/ directory
if not CSV_PATH.exists():
    CSV_PATH = pathlib.Path("../data/bgs_data/bgs_critical_minerals_production.csv")

print(f"Data file: {CSV_PATH}")
print(f"Exists   : {CSV_PATH.exists()}")

## 2. Data Preparation

We filter to **Production** records only, restrict to the most recent five years, and
aggregate by country × commodity (summing across sub-commodities / units to get a single
representative total).  Countries that produce fewer than two distinct commodities are dropped
because a single-commodity profile provides no meaningful basis for comparison.

For each surviving country we construct a short natural-language **production profile** — a
sentence the embedding model can sink its teeth into:

> *"Australia produces 12 minerals. Top production: iron ore: 930,000,000 tonnes;
> aluminium: 15,800,000 tonnes; ..."*

In [ ]:
# ── Load raw data ─────────────────────────────────────────────────────────────
df_raw = pd.read_csv(CSV_PATH, low_memory=False)
print(f"Raw rows : {len(df_raw):,}")
print(f"Columns  : {list(df_raw.columns)}")
df_raw.head(3)

In [ ]:
# ── Filter & clean ────────────────────────────────────────────────────────────
df = df_raw.copy()

# Keep production records only
df = df[df["statistic_type"].str.strip().str.lower() == "production"].copy()

# Coerce numeric columns
df["year"]     = pd.to_numeric(df["year"],     errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")

# Drop rows with missing key fields
df = df.dropna(subset=["year", "quantity", "country", "commodity"])
df = df[df["quantity"] > 0]

# Restrict to the most recent 5 years available in the dataset
max_year   = int(df["year"].max())
year_cutoff = max_year - 4          # inclusive window: [max_year-4, max_year]
df = df[df["year"] >= year_cutoff].copy()

print(f"Production rows after filter : {len(df):,}")
print(f"Year window                  : {year_cutoff} – {max_year}")
print(f"Unique countries             : {df['country'].nunique()}")
print(f"Unique commodities           : {df['commodity'].nunique()}")

In [ ]:
# ── Aggregate: country × commodity, mean quantity over the window ─────────────
# Using mean rather than sum avoids penalising countries with patchy reporting.
agg = (
    df.groupby(["country", "country_iso3", "commodity"], as_index=False)["quantity"]
    .mean()
    .rename(columns={"quantity": "mean_qty"})
)

# Drop countries that produce fewer than 2 distinct commodities
n_commodities = agg.groupby("country")["commodity"].nunique()
valid_countries = n_commodities[n_commodities >= 2].index
agg = agg[agg["country"].isin(valid_countries)].copy()

print(f"Countries with ≥2 commodities: {agg['country'].nunique()}")

In [ ]:
# ── Build natural-language profile for each country ───────────────────────────
TOP_N = 5   # how many top minerals to mention in the profile text

def build_profile(group: pd.DataFrame) -> pd.Series:
    """Given rows for one country, return a profile Series."""
    country   = group["country"].iloc[0]
    iso3      = group["country_iso3"].iloc[0] if "country_iso3" in group.columns else ""
    ranked    = group.sort_values("mean_qty", ascending=False)
    n_min     = len(ranked)
    total_qty = ranked["mean_qty"].sum()
    top_min   = ranked.iloc[0]["commodity"]

    top_items = ranked.head(TOP_N)
    top_str   = "; ".join(
        f"{row['commodity']}: {row['mean_qty']:,.0f} tonnes"
        for _, row in top_items.iterrows()
    )

    profile = (
        f"{country} produces {n_min} critical mineral{'s' if n_min != 1 else ''}. "
        f"Top production: {top_str}."
    )

    return pd.Series({
        "country"          : country,
        "iso3"             : iso3,
        "num_minerals"     : n_min,
        "total_production" : total_qty,
        "top_mineral"      : top_min,
        "profile_text"     : profile,
    })

profiles_df = (
    agg.groupby("country", group_keys=False)
    .apply(build_profile)
    .reset_index(drop=True)
)

print(f"Profile rows: {len(profiles_df)}")
profiles_df.head()

In [ ]:
# Preview a few profiles
for _, row in profiles_df.sample(5, random_state=42).iterrows():
    print(row["profile_text"])
    print()

## 3. Embedding Generation

**BAAI/bge-large-en-v1.5** is a 335M-parameter sentence encoder trained with contrastive
learning on large-scale text pairs.  Its 1024-dimensional output vectors are L2-normalised
before clustering so that cosine similarity is equivalent to dot product — a requirement for
HDBSCAN to behave well in high dimensions.

The model is downloaded automatically from HuggingFace Hub on first run (~1.3 GB) and cached
locally thereafter.  No internet access is needed for subsequent runs.

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
MODEL_NAME = "BAAI/bge-large-en-v1.5"
print(f"Loading embedding model: {MODEL_NAME} ...")
model = SentenceTransformer(MODEL_NAME)
print("Model loaded.")

In [ ]:
# ── Encode profiles ───────────────────────────────────────────────────────────
# BGE models benefit from an instruction prefix for retrieval / embedding tasks.
INSTRUCTION = "Represent the mineral production profile of this country for clustering: "

texts = [
    INSTRUCTION + text
    for text in profiles_df["profile_text"].tolist()
]

print(f"Encoding {len(texts)} country profiles ...")
embeddings_raw = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
)

# L2-normalise so cosine distance == Euclidean distance in unit-sphere space
embeddings = normalize(embeddings_raw, norm="l2")

print(f"Embedding matrix shape: {embeddings.shape}")
print(f"  • rows  = {embeddings.shape[0]} countries")
print(f"  • cols  = {embeddings.shape[1]} dimensions")

## 4. UMAP Dimensionality Reduction + HDBSCAN Clustering

**UMAP** (`n_neighbors=15`, `min_dist=0.1`) projects the 1024-D embeddings into a 2-D plane
that is used exclusively for visualisation.  Clustering is performed in the full 1024-D space
rather than on the 2-D projection to avoid information loss.

**HDBSCAN** (`min_cluster_size=3`, `min_samples=2`) identifies groups of countries with
similar production profiles and assigns a label of **-1** to noise/outlier countries that do
not belong to any cluster.

In [ ]:
# ── UMAP: 1024-D → 2-D for visualisation ─────────────────────────────────────
print("Running UMAP ...")
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="cosine",
    random_state=42,
)
umap_2d = reducer.fit_transform(embeddings)
print(f"UMAP output shape: {umap_2d.shape}")

In [ ]:
# ── HDBSCAN: cluster in full embedding space ──────────────────────────────────
# We cluster on the L2-normalised 1024-D embeddings with cosine metric to capture
# the full information content before projection.
print("Running HDBSCAN ...")
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=3,
    min_samples=2,
    metric="euclidean",   # equivalent to cosine on L2-normalised vectors
    cluster_selection_method="eom",
)
cluster_labels = clusterer.fit_predict(embeddings)

n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
n_noise    = int((cluster_labels == -1).sum())

print(f"Clusters found   : {n_clusters}")
print(f"Noise points     : {n_noise}  ({n_noise/len(cluster_labels)*100:.1f}% of countries)")

In [ ]:
# ── Assemble result DataFrame ─────────────────────────────────────────────────
result_df = profiles_df.copy()
result_df["umap_x"]  = umap_2d[:, 0]
result_df["umap_y"]  = umap_2d[:, 1]
result_df["cluster"] = cluster_labels
result_df["cluster_label"] = result_df["cluster"].apply(
    lambda c: f"Cluster {c}" if c >= 0 else "Noise"
)

result_df[["country", "cluster_label", "num_minerals", "top_mineral"]].head(10)

## 5. Interactive Cluster Visualisation

Each point represents a country.  Hover to see:
- **Country name**
- **Top mineral** — the commodity with the highest mean production over the window
- **Number of minerals** — breadth of the production portfolio
- **Total production** — approximate combined volume (all commodities, avg over 5 years)

Point size is proportional to total production, making major producers immediately visible.
Noise points (cluster = -1) are shown in grey.

In [ ]:
# ── Build colour palette ──────────────────────────────────────────────────────
import plotly.colors as pc

unique_labels = sorted(result_df["cluster_label"].unique())
cluster_only  = [l for l in unique_labels if l != "Noise"]

palette = pc.qualitative.Plotly + pc.qualitative.D3 + pc.qualitative.G10
colour_map = {label: palette[i % len(palette)] for i, label in enumerate(cluster_only)}
colour_map["Noise"] = "#cccccc"

# ── Scatter plot ──────────────────────────────────────────────────────────────
# Clamp bubble size so tiny producers are still visible
MAX_BUBBLE = 60
MIN_BUBBLE = 5

plot_df = result_df.copy()
log_prod = np.log1p(plot_df["total_production"])
# Normalise to [MIN_BUBBLE, MAX_BUBBLE]
plot_df["bubble_size"] = (
    (log_prod - log_prod.min()) / (log_prod.max() - log_prod.min() + 1e-9)
    * (MAX_BUBBLE - MIN_BUBBLE)
    + MIN_BUBBLE
)

fig = px.scatter(
    plot_df,
    x="umap_x",
    y="umap_y",
    color="cluster_label",
    color_discrete_map=colour_map,
    size="bubble_size",
    size_max=MAX_BUBBLE,
    hover_name="country",
    hover_data={
        "top_mineral"      : True,
        "num_minerals"     : True,
        "total_production" : ":.2e",
        "cluster_label"    : False,
        "umap_x"           : False,
        "umap_y"           : False,
        "bubble_size"      : False,
    },
    title="Country Mineral Production Clusters (UMAP + HDBSCAN on BGE-Large Embeddings)",
    labels={
        "cluster_label"   : "Cluster",
        "umap_x"          : "UMAP Dimension 1",
        "umap_y"          : "UMAP Dimension 2",
    },
    template="plotly_white",
    width=950,
    height=650,
)

fig.update_layout(
    legend=dict(title="Cluster", itemsizing="constant"),
    font=dict(size=13),
)
fig.show()

## 6. Cluster Interpretation

For each cluster we print:
- **Member countries**
- **Dominant minerals** — commodities that appear most frequently as the top mineral
- **Average portfolio breadth** — mean number of distinct commodities per country

In [ ]:
cluster_ids = sorted(set(cluster_labels))

for cid in cluster_ids:
    subset = result_df[result_df["cluster"] == cid]
    label  = "NOISE" if cid == -1 else f"CLUSTER {cid}"
    countries    = ", ".join(sorted(subset["country"].tolist()))
    top_minerals = (
        subset["top_mineral"]
        .value_counts()
        .head(3)
        .index.tolist()
    )
    avg_minerals = subset["num_minerals"].mean()

    print(f"{'='*70}")
    print(f"  {label}  ({len(subset)} countries)")
    print(f"  Countries        : {countries}")
    print(f"  Dominant minerals: {', '.join(top_minerals)}")
    print(f"  Avg minerals/country: {avg_minerals:.1f}")
    print()

## 7. Similarity Search

Given a **query country**, we compute the cosine similarity between its embedding and all
other country embeddings and return the top-10 most similar nations.

This is useful for supply-chain analysis: if a query country experiences a production
disruption, the most similar countries represent the closest substitutes.

In [ ]:
def find_similar_countries(
    query_country: str,
    profiles: pd.DataFrame,
    emb_matrix: np.ndarray,
    top_k: int = 10,
) -> pd.DataFrame:
    """
    Return the `top_k` countries most similar to `query_country` based on
    cosine similarity of their production-profile embeddings.
    """
    # Locate query index (case-insensitive)
    mask = profiles["country"].str.lower() == query_country.lower()
    if mask.sum() == 0:
        # Try partial match
        mask = profiles["country"].str.lower().str.contains(query_country.lower())
    if mask.sum() == 0:
        available = profiles["country"].tolist()
        raise ValueError(
            f"'{query_country}' not found in profiles.\n"
            f"Available countries (first 20): {available[:20]}"
        )

    idx       = profiles[mask].index[0]
    query_emb = emb_matrix[idx].reshape(1, -1)

    sims = cosine_similarity(query_emb, emb_matrix).flatten()

    sim_df = profiles.copy()
    sim_df["similarity"] = sims

    # Exclude the query country itself
    sim_df = sim_df[sim_df.index != idx]
    sim_df = sim_df.sort_values("similarity", ascending=False).head(top_k)

    query_name = profiles.loc[idx, "country"]
    print(f"Top {top_k} countries most similar to '{query_name}':\n")
    print(
        sim_df[["country", "top_mineral", "num_minerals", "total_production", "similarity"]]
        .to_string(index=False)
    )
    return sim_df


# ── Run for the default query country ─────────────────────────────────────────
QUERY_COUNTRY = "China"
similar_df = find_similar_countries(QUERY_COUNTRY, profiles_df, embeddings)

In [ ]:
# ── Visualise similarity scores ────────────────────────────────────────────────
fig_sim = px.bar(
    similar_df.sort_values("similarity"),
    x="similarity",
    y="country",
    orientation="h",
    color="similarity",
    color_continuous_scale="Blues",
    hover_data=["top_mineral", "num_minerals", "total_production"],
    title=f"Top 10 Countries Most Similar to {QUERY_COUNTRY} (Cosine Similarity)",
    labels={"similarity": "Cosine Similarity", "country": "Country"},
    template="plotly_white",
    width=750,
    height=450,
)
fig_sim.update_coloraxes(showscale=False)
fig_sim.show()

## 8. Bonus — Try Your Own Query Country

Change `QUERY_COUNTRY` in the cell below to any country in the dataset to explore its nearest
neighbours in production-profile space.

In [ ]:
# ── Interactive query — change this country name ──────────────────────────────
QUERY_COUNTRY_2 = "United States of America"   # <-- edit me

try:
    _ = find_similar_countries(QUERY_COUNTRY_2, profiles_df, embeddings)
except ValueError as e:
    print(e)

In [ ]:
# ── Print all available countries for reference ────────────────────────────────
print("All countries in the clustering dataset:")
print(", ".join(sorted(profiles_df["country"].tolist())))

---

## Summary

| Step | Tool | Purpose |
|------|------|---------|
| Profile construction | pandas | Convert tabular production data into natural-language text |
| Embedding | BAAI/bge-large-en-v1.5 (1024-D) | Encode semantic meaning of production profiles |
| Dimensionality reduction | UMAP | Project to 2-D for human-readable visualisation |
| Clustering | HDBSCAN | Discover country groups without a fixed cluster count |
| Similarity search | cosine_similarity | Find substitute producers for supply-chain risk |

**Key insights to explore:**
- Do geographically distant countries cluster together because of similar production mixes?
- Which countries are consistent noise outliers with unique production signatures?
- How does the similarity landscape change if you restrict to a single commodity group?